In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/d2c_data/d2c churn data package/rfm_modeling_snapshot.csv")

print(df.shape)
display(df.head())

df.info()

(2400, 29)


,customer_id,snapshot_date,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,marketing_consent,recency_days,frequency_180d,...,sessions_30d,product_views_30d,cart_adds_30d,wishlist_adds_30d,abandoned_carts_30d,email_opens_30d,campaign_clicks_30d,last_visit_days_ago,churn_next_60d,split
0,CUST00001,2025-09-30,Tier 1,18-24,Instagram,Silver,Makeup,Yes,107,1,...,1,4,0,0,0,2,0,20,1,train
1,CUST00002,2025-09-30,Tier 2,25-34,Marketplace,Silver,Hair Care,Yes,40,1,...,8,31,4,2,3,0,0,0,0,train
2,CUST00003,2025-09-30,Tier 1,25-34,Influencer,NaN,Skin Care,Yes,171,1,...,1,3,0,0,0,0,0,26,1,train
3,CUST00004,2025-09-30,Tier 3,25-34,Google Search,NaN,Fragrance,No,131,1,...,1,6,0,0,0,0,0,14,1,train
4,CUST00005,2025-09-30,Tier 3,35-44,Organic,Gold,Hair Care,Yes,38,3,...,18,95,4,1,1,3,1,9,0,train


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               2400 non-null   object 
 1   snapshot_date             2400 non-null   object 
 2   city_tier                 2400 non-null   object 
 3   age_group                 2400 non-null   object 
 4   acquisition_channel       2400 non-null   object 
 5   loyalty_tier              1014 non-null   object 
 6   preferred_category        2400 non-null   object 
 7   marketing_consent         2400 non-null   object 
 8   recency_days              2400 non-null   int64  
 9   frequency_180d            2400 non-null   int64  
 10  monetary_180d             2400 non-null   float64
 11  return_rate_180d          2400 non-null   float64
 12  avg_discount_pct_180d     2400 non-null   float64
 13  avg_rating_180d           2400 non-null   float64
 14  category

In [9]:
df.columns.tolist()

['customer_id',
 'snapshot_date',
 'city_tier',
 'age_group',
 'acquisition_channel',
 'loyalty_tier',
 'preferred_category',
 'marketing_consent',
 'recency_days',
 'frequency_180d',
 'monetary_180d',
 'return_rate_180d',
 'avg_discount_pct_180d',
 'avg_rating_180d',
 'category_diversity_180d',
 'ticket_count_90d',
 'negative_ticket_rate_90d',
 'avg_resolution_hours_90d',
 'days_since_signup',
 'sessions_30d',
 'product_views_30d',
 'cart_adds_30d',
 'wishlist_adds_30d',
 'abandoned_carts_30d',
 'email_opens_30d',
 'campaign_clicks_30d',
 'last_visit_days_ago',
 'churn_next_60d',
 'split']

In [10]:
df['split'].value_counts()

,count
split,
train,1728
validation,336
test,336


In [11]:
df['churn_next_60d'].value_counts()

,count
churn_next_60d,
0,1273
1,1127


In [12]:
df['churn_next_60d'].value_counts(normalize=True)

,proportion
churn_next_60d,
0,0.530417
1,0.469583


In [13]:
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'valid']
test_df  = df[df['split'] == 'test']

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1728, 29)
(0, 29)
(336, 29)


In [16]:
train_df = df[df['split'] == 'train']
val_df   = df[df['split'] == 'validation']
test_df  = df[df['split'] == 'test']

In [17]:
print(df['split'].unique())
print(df['split'].value_counts())

['train' 'validation' 'test']
split
train         1728
validation     336
test           336
Name: count, dtype: int64


In [18]:
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'validation']
test_df = df[df['split'] == 'test']

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1728, 29)
(336, 29)
(336, 29)


In [19]:
target = 'churn_next_60d'

drop_cols = [
    'customer_id',
    'snapshot_date',
    'split',
    target
]

X_train = train_df.drop(columns=drop_cols)
X_val = val_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

y_train = train_df[target]
y_val = val_df[target]
y_test = test_df[target]

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_cols = X_train.select_dtypes(include='object').columns
num_cols = X_train.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=2000))
])

log_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier',
       'preferred_category', 'marketing_consent'],
      dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
       'avg_discount_pct_...ing_180d', 'category_diversity_180d',
       'ticket_count_90d', 'negative_ticket_rate_90d',
       'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d',
       'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d',
       'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d',
       'last_visit_days_ago'],
      dtype='object'))])),
                ('model', LogisticRegression(max_iter=2000))])

In [22]:
from sklearn.metrics import *

log_probs = log_model.predict_proba(X_test)[:,1]

print("ROC-AUC:",
      roc_auc_score(y_test, log_probs))

ROC-AUC: 0.8870110544217686


In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model',
     RandomForestClassifier(
         n_estimators=300,
         max_depth=10,
         random_state=42
     ))
])

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier',
       'preferred_category', 'marketing_consent'],
      dtype='object')),
                                                 ('num', 'passthrough',
                                                  Index(['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
       'avg_discount_pct_...
       'ticket_count_90d', 'negative_ticket_rate_90d',
       'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d',
       'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d',
       'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d',
       'last_visit_days_ago'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(max_depth=10, n_estimators=300,
                                        random_state=42))])

In [24]:
rf_probs = rf_model.predict_proba(X_test)[:,1]

print(
    "RF ROC-AUC:",
    roc_auc_score(y_test, rf_probs)
)

RF ROC-AUC: 0.8847789115646258


In [25]:
threshold = 0.4

log_preds = (log_probs >= threshold).astype(int)

from sklearn.metrics import *

accuracy = accuracy_score(y_test, log_preds)
precision = precision_score(y_test, log_preds)
recall = recall_score(y_test, log_preds)
f1 = f1_score(y_test, log_preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc_score(y_test, log_probs))

Accuracy: 0.8154761904761905
Precision: 0.7819148936170213
Recall: 0.875
F1: 0.8258426966292135
ROC-AUC: 0.8870110544217686


###Feature Importance

In [26]:
feature_names = (
    log_model.named_steps['preprocessor']
    .get_feature_names_out()
)

coefficients = (
    log_model.named_steps['model']
    .coef_[0]
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

importance_df["abs_coef"] = (
    importance_df["coefficient"]
    .abs()
)

importance_df = (
    importance_df
    .sort_values(
        "abs_coef",
        ascending=False
    )
)

importance_df.head(20)

,feature,coefficient,abs_coef
28,num__return_rate_180d,1.504550,1.504550
29,num__avg_discount_pct_180d,0.860855,0.860855
33,num__negative_ticket_rate_90d,0.852196,0.852196
32,num__ticket_count_90d,-0.552606,0.552606
18,cat__preferred_category_Fragrance,-0.441940,0.441940
14,cat__loyalty_tier_Platinum,-0.423532,0.423532
11,cat__acquisition_channel_Organic,-0.395488,0.395488
24,cat__marketing_consent_Yes,-0.306895,0.306895
42,num__campaign_clicks_30d,-0.185365,0.185365
31,num__category_diversity_180d,0.181206,0.181206


In [28]:
results = X_test.copy()

results["customer_id"] = test_df["customer_id"].values
results["actual"] = y_test.values
results["predicted"] = log_preds
results["probability"] = log_probs

In [29]:
false_pos = results[
    (results.actual == 0)
    & (results.predicted == 1)
]

false_pos[
    ["customer_id","probability"]
].head(5)

,customer_id,probability
43,CUST00044,0.406530
108,CUST00109,0.504232
247,CUST00248,0.509545
334,CUST00335,0.758473
436,CUST00437,0.927427


In [30]:
false_neg = results[
    (results.actual == 1)
    & (results.predicted == 0)
]

false_neg[
    ["customer_id","probability"]
].head(5)

,customer_id,probability
87,CUST00088,0.366904
183,CUST00184,0.062165
246,CUST00247,0.280786
567,CUST00568,0.388968
591,CUST00592,0.232213


##
### Business Interpretation of Feature Importance

The model suggests that customer dissatisfaction and weak engagement are major drivers of churn.

Customers exhibiting high return rates, negative support interactions, and heavy discount dependence show elevated churn risk.

Conversely, loyalty program participation, campaign engagement, and consistent purchasing behavior are associated with stronger retention.

These findings support targeted retention strategies focused on customer experience improvement, proactive support outreach, and loyalty engagement.